In [1]:
import pandas as pd
from pprint import pprint
import json
import re
from typing import Dict, List, Any

In [6]:
def load_mixed_json(file_path):
    objs = []
    decoder = json.JSONDecoder()

    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    idx = 0
    while idx < len(content):
        # Skip whitespace
        while idx < len(content) and content[idx].isspace():
            idx += 1
        if idx >= len(content):
            break

        # Decode next JSON entity
        obj, end = decoder.raw_decode(content, idx)

        # Normalize: always append as list of dicts
        if isinstance(obj, dict):
            objs.append(obj)
        elif isinstance(obj, list):
            objs.extend(obj)
        else:
            raise ValueError(f"Unsupported JSON structure at index {idx}")

        idx = end

    return objs


In [19]:
json_list1 = load_mixed_json("stage2/gemini_data1.json")
json_list2 = load_mixed_json("stage2/gemini_data2.json")
json_list3 = load_mixed_json("stage2/gemini_data3.json")
json_list4 = load_mixed_json("stage2/gemini_data4.json")

In [20]:
print(len(json_list1))
print(len(json_list2))
print(len(json_list3))
print(len(json_list4))

55
132
210
67


In [21]:
def extract_unique_goals(samples):
    """
    Extract unique values of goal_category and goal_type from a list of samples.

    Args:
        samples (list[dict]): List of sample dictionaries.

    Returns:
        dict: Dictionary with unique goal_category and goal_type values.
    """
    goal_categories = set()
    goal_types = set()

    for sample in samples:
        if "goal_category" in sample:
            goal_categories.add(sample["goal_category"])
        if "goal_type" in sample:
            goal_types.add(sample["goal_type"])

    return {
        "unique_goal_categories": sorted(goal_categories),
        "unique_goal_types": sorted(goal_types),
    }


In [29]:
def normalize_goal_types(samples):
    """
    Normalize goal_type in each sample to one of:
    learning, work, personal, health.

    Args:
        samples (list[dict]): List of sample dictionaries.

    Returns:
        list[dict]: Updated list of samples with normalized goal_type.
    """
    for sample in samples:
        goal_type = sample.get("goal_type", "").strip()
        g_lower = goal_type.lower()

        # Learning-related
        if any(
            keyword in g_lower
            for keyword in [
                "academic",
                "study",
                "exam",
                "education",
                "learning",
                "skill",
                "knowledge",
                "certification",
                "training",
                "algorithm",
                "data structure",
                "computer",
                "programming",
                "system design",
                "sql",
                "cloud",
                "network",
                "engineering",
            ]
        ):
            sample["goal_type"] = "learning"

        # Work-related
        elif any(
            keyword in g_lower
            for keyword in [
                "career",
                "professional",
                "productivity",
                "organization",
                "work",
                "devops",
                "operations",
                "project",
                "performance",
                "software",
                "architecture",
                "administration",
            ]
        ):
            sample["goal_type"] = "work"

        # Personal-related
        elif any(
            keyword in g_lower
            for keyword in [
                "personal",
                "relationships",
                "social",
                "community",
                "creative",
                "lifestyle",
                "home",
                "parenting",
                "cultural",
                "transition",
                "hobby",
            ]
        ):
            sample["goal_type"] = "personal"

        # Health-related
        elif any(
            keyword in g_lower
            for keyword in [
                "health",
                "well-being",
                "fitness",
                "nutrition",
                "diet",
                "sleep",
                "therapy",
                "recovery",
                "mental",
                "mindfulness",
                "motor",
                "physical",
            ]
        ):
            sample["goal_type"] = "health"

        else:
            # Default fallback
            sample["goal_type"] = "personal"

    return samples

In [23]:
extract_unique_goals(json_list1+json_list2+json_list3+json_list4)

{'unique_goal_categories': ['habit', 'one_time'],
 'unique_goal_types': ['Academic Requirement',
  'Advanced Computer Science',
  'Advanced Data Structures',
  'Advanced Engineering',
  'Advanced SQL',
  'Algorithm Implementation',
  'Algorithm Mastery',
  'Algorithm Optimization',
  'Algorithm Patterns',
  'Algorithm Selection',
  'Algorithm Visualization',
  'Algorithms & Data Structures',
  'Algorithms & Visualization',
  'Career Growth',
  'Career Preparation',
  'Career Transition',
  'Career/Digital Wellness',
  'Certification Study',
  'Civic Engagement',
  'Cloud Architecture',
  'Cloud Networking',
  'Cloud Tooling',
  'Cognitive Tools',
  'Communication',
  'Community & Collaboration',
  'Composite Data Structures',
  'Computer Architecture',
  'Computer Graphics',
  'Creative Growth',
  'Creative/Hobby',
  'Creative/Mental Health',
  'Cyber Security',
  'Cybersecurity',
  'Data Engineering',
  'Data Engineering & Algorithms',
  'Data Modeling',
  'Data Structure Implementati

In [30]:
mapped = normalize_goal_types(json_list1 + json_list2 + json_list3 + json_list4)

In [47]:
def validate_samples_structure(samples):
    """
    Validate that all samples in the list follow the required structure.
    Checks for missing keys and extra keys.

    Args:
        samples (list[dict]): List of sample dictionaries.

    Returns:
        bool: True if all samples are valid, False otherwise.
        list: List of error messages for invalid samples.
    """
    required_top_keys = {
        "goal", "goal_category", "goal_type", "time_horizon",
        "description", "baseline", "tasks"
    }
    required_baseline_keys = {
        "fixed_constraints", "physical_constraints", "non_negotiables"
    }
    required_task_keys = {
        "index", "task", "description", "is_repeatable",
        "repeat_frequency", "gap_flag", "estimated_duration_minutes"
    }
    required_duration_keys = {"min", "max"}

    errors = []

    for i, sample in enumerate(samples):
        # --- Top-level keys ---
        missing_top = required_top_keys - sample.keys()
        extra_top = sample.keys() - required_top_keys
        if missing_top:
            errors.append(f"Sample {i} missing top-level keys: {missing_top}")
        if extra_top:
            errors.append(f"Sample {i} has extra top-level keys: {extra_top}")

        # --- Baseline ---
        baseline = sample.get("baseline")
        if not isinstance(baseline, dict):
            errors.append(f"Sample {i} baseline is not a dict")
        else:
            missing_baseline = required_baseline_keys - baseline.keys()
            extra_baseline = baseline.keys() - required_baseline_keys
            if missing_baseline:
                errors.append(f"Sample {i} baseline missing keys: {missing_baseline}")
            if extra_baseline:
                errors.append(f"Sample {i} baseline has extra keys: {extra_baseline}")

        # --- Tasks ---
        tasks = sample.get("tasks")
        if not isinstance(tasks, list):
            errors.append(f"Sample {i} tasks is not a list")
        else:
            for j, task in enumerate(tasks):
                missing_task = required_task_keys - task.keys()
                extra_task = task.keys() - required_task_keys
                if missing_task:
                    errors.append(f"Sample {i} task {j} missing keys: {missing_task}")
                if extra_task:
                    errors.append(f"Sample {i} task {j} has extra keys: {extra_task}")

                # --- Duration ---
                duration = task.get("estimated_duration_minutes")
                if not isinstance(duration, dict):
                    errors.append(f"Sample {i} task {j} duration is not a dict")
                else:
                    missing_duration = required_duration_keys - duration.keys()
                    extra_duration = duration.keys() - required_duration_keys
                    if missing_duration:
                        errors.append(f"Sample {i} task {j} duration missing keys: {missing_duration}")
                    if extra_duration:
                        errors.append(f"Sample {i} task {j} duration has extra keys: {extra_duration}")

    return len(errors) == 0, errors




In [53]:
def convert_to_target_schema(samples):
    """
    Convert a list of samples into the target dataset schema format.

    Args:
        samples (list[dict]): List of sample dictionaries.

    Returns:
        list[dict]: Converted samples following the target schema.
    """
    converted = []

    for sample in samples:
        new_sample = {
            "goal": sample.get("goal", ""),
            "goal_category": sample.get("goal_category", ""),
            "goal_type": sample.get("goal_type", ""),
            "time_horizon": int(sample.get("time_horizon", 0)),
            "description": sample.get("description", ""),
            "baseline": {
                "fixed_constraints": sample.get("baseline", {}).get(
                    "fixed_constraints", ""
                ),
                "physical_constraints": sample.get("baseline", {}).get(
                    "physical_constraints", ""
                ),
                "non_negotiables": sample.get("baseline", {}).get(
                    "non_negotiables", ""
                ),
            },
            "tasks": [],
        }

        # Transform tasks into the target schema
        for j, task in enumerate(sample.get("tasks", [])):
            new_task = {
                "index": task.get("index", j),  # fallback to enumeration index
                "task": task.get("task", ""),
                "description": task.get("description", ""),
                "is_repeatable": task.get("is_repeatable", False),
                "repeat_frequency": task.get("repeat_frequency", 0),
                "gap_flag": task.get("gap_flag", False),
                "estimated_duration_minutes": {
                    "min": task.get("estimated_duration_minutes", {}).get("min", 0),
                    "max": task.get("estimated_duration_minutes", {}).get("max", 0),
                },
            }
            new_sample["tasks"].append(new_task)

        converted.append(new_sample)

    return converted


In [54]:
list1_converted=convert_to_target_schema(json_list1)

In [57]:
pprint(list1_converted[0])

{'baseline': {'fixed_constraints': 'Working full-time 40 hours per week; study '
                                   'limited to evenings and weekends.',
              'non_negotiables': 'Must achieve >70% in all Ethics practice '
                                 'modules before taking the final mock.',
              'physical_constraints': 'Suffer from eye fatigue; requires Blue '
                                      'Light filters and 20-20-20 rule '
                                      'breaks.'},
 'description': 'Complete the curriculum and practice questions for the CFA '
                'Level I exam, focusing on high-weight areas like Ethics and '
                'Financial Statement Analysis.',
 'goal': 'CFA Level I Exam - Core Content Mastery',
 'goal_category': 'one_time',
 'goal_type': 'learning',
 'tasks': [{'description': 'Sign up for the exam and install the CFA Institute '
                           'candidate app and curriculum files.',
            'estimated_duration_

In [62]:
validate_samples_structure(converted_mapped)

(True, [])

In [58]:
converted_mapped=convert_to_target_schema(mapped)

In [61]:
count=0
for sample in converted_mapped:
    for task in sample.get("tasks",[]):
        if task.get("estimated_duration_minutes",{}).get("min",0)>30:
            count+=1

print(count)
print(len(converted_mapped))

237
464


In [63]:
def save_for_finetune(converted, filename="converted_dataset.jsonl"):
    """
    Save the converted dataset in JSONL format for LLM fine-tuning.

    Args:
        converted (list): List of schema dicts produced by preprocess_dataset.
        filename (str): Output file name.
    """
    with open(filename, "w", encoding="utf-8") as f:
        for sample in converted:
            # Each sample becomes one JSON line
            f.write(json.dumps(sample, ensure_ascii=False) + "\n")

In [64]:
save_for_finetune(converted_mapped,filename="final/gemini_dataset.jsonl")